In [1]:
import xarray as xr
import numpy as np
import xesmf as xe
from dask.diagnostics import ProgressBar
import json

In [2]:
cmems_phy1 = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-mob_glo_phy_my_0.125deg_P1M-m.zarr", consolidated=True)
cmems_phy2 = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-mob_glo_phy_nrt_0.125deg_P1M-m.zarr", consolidated=True)

In [3]:
# 1. Find the maximum (latest) time in the first dataset
max_time_phy1 = cmems_phy1.time.max()

# 2. Extract only the times from phy2 that occur strictly AFTER phy1 ends.
# This prevents duplication and effectively prioritizes phy1 for the overlap period.
valid_times_for_phy2 = cmems_phy2.time[cmems_phy2.time > max_time_phy1]
cmems_phy2_filtered = cmems_phy2.sel(time=valid_times_for_phy2)

# 3. Concatenate the datasets along the time dimension
# (The 'if' statement prevents errors just in case phy1 already completely covers phy2's timespan)
if cmems_phy2_filtered.sizes['time'] > 0:
    cmems_combined = xr.concat([cmems_phy1, cmems_phy2_filtered], dim="time")
else:
    cmems_combined = cmems_phy1

# 4. (Optional but recommended) Ensure the resulting time dimension is perfectly sorted
cmems_combined = cmems_combined.sortby('time')
cmems_phy = cmems_combined

In [4]:

lat_diff = cmems_phy['latitude'].diff(dim='latitude')
lat_diff_vals = lat_diff.compute()
lon_diff = cmems_phy['longitude'].diff(dim='longitude')
lon_diff_vals = lon_diff.compute()
print("Latitude spacing unique values:", np.unique(lat_diff_vals))
print("Longitude spacing unique values:", np.unique(lon_diff_vals))
print("Bounds")
print(cmems_phy['latitude'].min().compute().item(), cmems_phy['latitude'].max().compute().item())
print(cmems_phy['longitude'].min().compute().item(), cmems_phy['longitude'].max().compute().item())

Latitude spacing unique values: [0.125]
Longitude spacing unique values: [0.125]
Bounds
-56.9375 -40.0625
-65.9375 -52.0625


In [5]:
#now regrid to regular grid for the model
with open("./data/bbox.json", "r") as f:
    bbox = json.load(f)
min_lon, min_lat, max_lon, max_lat = [bbox["min_lon"], bbox["min_lat"], bbox["max_lon"], bbox["max_lat"]]

res = 0.125
new_lons = np.arange(min_lon, max_lon + res, res)
new_lats = np.arange(min_lat, max_lat + res, res)

# Create a target grid as xarray Dataset
ds_tgt = xr.Dataset({
    'lat': (['lat'], new_lats),
    'lon': (['lon'], new_lons)})

regridder = xe.Regridder(cmems_phy, ds_tgt, 'conservative')
with ProgressBar():
    cmems_phy_regridded = regridder(cmems_phy).compute()


[########################################] | 100% Completed | 38.43 ss


In [6]:
lat_diff = cmems_phy_regridded['lat'].diff(dim='lat')
lat_diff_vals = lat_diff.compute()
lon_diff = cmems_phy_regridded['lon'].diff(dim='lon')
lon_diff_vals = lon_diff.compute()
print("Latitude spacing unique values:", np.unique(lat_diff_vals))
print("Longitude spacing unique values:", np.unique(lon_diff_vals))
print("Regrided bounds:")
print(cmems_phy_regridded['lat'].min().compute().item(), cmems_phy_regridded['lat'].max().compute().item())
print(cmems_phy_regridded['lon'].min().compute().item(), cmems_phy_regridded['lon'].max().compute().item())

Latitude spacing unique values: [0.125]
Longitude spacing unique values: [0.125]
Regrided bounds:
-57.0 -40.0
-66.0 -52.0


In [7]:
cmems_phy_regridded.to_zarr("./resources/copernicus_marine_service/regridded/obs-mob_glo_phy_regridded.zarr", consolidated=True, mode="w")
